# bhavkit as a Python package

`bhavkit` is both a CLI and a normal Python package. This workbook shows the **library API**
("use it as a uv package") end to end: create a DuckDB database, push real bhavcopy files
through the cleaning pipeline, run template queries, build a data-quality report, export to
parquet, and drive the CLI from Python.

No live NSE access is needed — everything runs against scratch files and a local database
created in a temp directory. Expected runtime: a few seconds.

## Prerequisites

* Python 3.11+ and [uv](https://docs.astral.sh/uv)
* The package available in this environment. From the repo root:

```bash
uv sync                          # installs the editable package + dev tools
uv run --with ipykernel \
    jupyter notebook workbook    # start Jupyter against that env
```

In any other uv project, add it as a dependency instead:

```bash
uv add bhavkit                 # from PyPI / a git ref
uv add --editable ../bhavkit   # from a local checkout
```

In [1]:
import sys
from pathlib import Path

import bhavkit

print("bhavkit", bhavkit.__version__)
print("python", sys.version.split()[0])
print("module file:", Path(bhavkit.__file__).resolve())

bhavkit 0.1.0
python 3.12.13
module file: /Users/shramadeep/Documents/Personel_projects/bhavkit/src/bhavkit/__init__.py


## 1. Create a database with `Database`

This is the same code the CLI runs for `bhavkit init`.

In [2]:
import tempfile
from datetime import date
from pathlib import Path

from bhavkit.db import Database

WORK = Path(tempfile.mkdtemp(prefix="bhavkit-nb-"))
print("scratch dir:", WORK)

db = Database(WORK / "bhavkit.duckdb")
db.bootstrap()
print("schema version:", db.schema_version)

for row in db.query("SHOW TABLES").iter_rows():
    print("table:", row[0])

scratch dir: /var/folders/5b/zkr9qxl525x_3_cnh0gfh7600000gn/T/bhavkit-nb-ai27tt0r
schema version: 2
table: bhav_daily
table: data_gaps
table: deliverable_daily
table: equity_master
table: fo_daily
table: index_daily
table: ingest_log
table: qc_issues
table: schema_migrations


## 2. Ingest a bhavcopy file through the real pipeline

`read_bhav_file` -> `clean_bars` -> `Database.upsert_bars` is exactly what the
`bhavkit update` command runs per file. We write a realistic CM bhavcopy CSV for two
trading days and push both through the pipeline, including QC issues and the ingest audit log.

In [3]:
from bhavkit.clean import clean_bars, read_bhav_file

HEADER = (
    "SYMBOL, SERIES,DATE, PREV_CLOSE,OPEN_PRICE,HIGH_PRICE,LOW_PRICE,LAST_PRICE,"
    "CLOSE_PRICE,AVG_PRICE,TTL_TRD_QNTY,TURNOVER_LACS,NO_OF_TRADES,DELIV_QTY,DELIV_PER"
)


def rows_for(day: date) -> list[str]:
    d = day.strftime("%d-%b-%Y").upper()
    return [
        f"SBIN,EQ,{d},620.00,626.00,635.50,621.10,631.00,632.00,628.50,4420123,278012.45,185432,1000000,84.32",
        f"TATASTEEL,EQ,{d},132.00,133.20,136.00,132.10,135.10,135.50,134.40,32100000,410223.10,254311,25000000,78.10",
        f"HDFCBANK,EQ,{d},1654.00,1658.00,1690.00,1650.00,1685.00,1688.00,1672.00,9800000,1600000.50,221100,4500000,82.25",
    ]


for day in (date(2024, 1, 3), date(2024, 1, 4)):
    name = f"cm{day.strftime('%d%b%Y').upper()}bhav.csv"
    csv_path = WORK / name
    csv_path.write_text(HEADER + "\n" + "\n".join(rows_for(day)) + "\n")

    raw = read_bhav_file(csv_path)
    bars, issues = clean_bars(raw, expected_day=day)
    inserted = db.upsert_bars(bars, source_file=name)
    db.write_qc_issues("cm", day, issues)
    db.write_ingest_log(
        product="cm", date=day, status="ingested",
        source_url=csv_path.as_uri(), file_name=name, num_rows=inserted,
    )
    print(f"{day}: {inserted} rows, {len(issues)} qc issues")

print("bars total:", db.fetchone("SELECT count(*) FROM bhav_daily"))

2024-01-03: 3 rows, 0 qc issues
2024-01-04: 3 rows, 0 qc issues
bars total: (6,)


## 3. Ad-hoc SQL + template queries

`Database.query` / `Database.fetchone` run arbitrary SQL and hand back a Polars frame
or a tuple. `bhavkit.query` wraps this in a read-only helper plus built-in templates.

In [4]:
from bhavkit.query import execute_template, run_sql

df = run_sql(db, "SELECT symbol, date, close FROM bhav_daily ORDER BY date, symbol")
print(df.head())

top = execute_template(db, "top_gainers")
assert top is not None
print(top)

shape: (5, 3)
┌───────────┬────────────┬────────┐
│ symbol    ┆ date       ┆ close  │
│ ---       ┆ ---        ┆ ---    │
│ str       ┆ date       ┆ f64    │
╞═══════════╪════════════╪════════╡
│ HDFCBANK  ┆ 2024-01-03 ┆ 1688.0 │
│ SBIN      ┆ 2024-01-03 ┆ 632.0  │
│ TATASTEEL ┆ 2024-01-03 ┆ 135.5  │
│ HDFCBANK  ┆ 2024-01-04 ┆ 1688.0 │
│ SBIN      ┆ 2024-01-04 ┆ 632.0  │
└───────────┴────────────┴────────┘
shape: (3, 5)
┌───────────┬────────────┬────────┬────────────┬──────┐
│ symbol    ┆ date       ┆ close  ┆ prev_close ┆ pct  │
│ ---       ┆ ---        ┆ ---    ┆ ---        ┆ ---  │
│ str       ┆ date       ┆ f64    ┆ f64        ┆ f64  │
╞═══════════╪════════════╪════════╪════════════╪══════╡
│ TATASTEEL ┆ 2024-01-04 ┆ 135.5  ┆ 132.0      ┆ 2.65 │
│ HDFCBANK  ┆ 2024-01-04 ┆ 1688.0 ┆ 1654.0     ┆ 2.06 │
│ SBIN      ┆ 2024-01-04 ┆ 632.0  ┆ 620.0      ┆ 1.94 │
└───────────┴────────────┴────────┴────────────┴──────┘


## 4. Data-quality report, programmatically

`build_report` produces the same `Report` object the CLI serialises to Markdown + JSON.

In [5]:
from bhavkit.report import build_report, refresh_gaps_table, to_json, to_markdown

rep = build_report(db, "cm", date(2024, 1, 1), date(2024, 1, 31))
gap_rows = refresh_gaps_table(db, rep)

print(to_markdown(rep))
print("\ndata_gaps rows written:", gap_rows)

(WORK / "report_cm.md").write_text(to_markdown(rep))
(WORK / "report_cm.json").write_text(to_json(rep))

# bhavkit data-quality report — cm

- generated: 2026-09-19T00:24:14
- range: 2024-01-01 .. 2024-01-31
- gaps: 19 missing, 0 errors, 0 unknown holidays

## Monthly coverage

| month | expected | data days | holidays | errors | coverage |
| ------ | -------: | --------: | -------: | -----: | -------: |
| 2024-01 | 21 | 2 | 0 | 0 | 9.52% |

## Gaps

Missing attempts (weekdays with no record):
 - 2024-01-01, 2024-01-02, 2024-01-05, 2024-01-08, 2024-01-09, 2024-01-10, 2024-01-11, 2024-01-12, 2024-01-15, 2024-01-16, 2024-01-17, 2024-01-18, 2024-01-19, 2024-01-23, 2024-01-24, 2024-01-25, 2024-01-29, 2024-01-30, 2024-01-31

## Anomalies

### EXTREME_MOVE (0)

### ZERO_VOLUME_WITH_TRADES (0)

### NEGATIVE_PRICE (0)

### DELIVERY_MISMATCH (0)

## Equity master drift

none

## Cross-dataset coverage

| table | days | rows | first | last |
| --- | ---: | ---: | --- | --- |
| index_daily | 0 | 0 | - | - |
| fo_daily | 0 | 0 | - | - |
| deliverable_daily | 0 | 0 | - | - |


data_gaps rows written: 

1622

## 5. Export to parquet / CSV

`export_table` wraps DuckDB's `COPY TO` with the same filters as `bhavkit export`.

In [6]:
import polars as pl

from bhavkit.export import export_table

out_file = export_table(db, "bhav_daily", out=WORK / "bhav.parquet", fmt="parquet")
print("exported:", out_file)
back = pl.read_parquet(out_file)
print("rows read back:", back.shape)

exported: /private/var/folders/5b/zkr9qxl525x_3_cnh0gfh7600000gn/T/bhavkit-nb-ai27tt0r/bhav.parquet
rows read back: (6, 17)


## 6. The same thing as a CLI, from Python

The console entry point is `bhavkit.cli:app`, so `python -m bhavkit.cli` works
wherever the package is importable (no shell `bhavkit` executable needed).

In [7]:
import os
import subprocess
import sys

# DuckDB takes a file lock while a connection is open, so release the
# in-process handle before handing the same DB file to the CLI process.
db.close()

proc = subprocess.run(
    [sys.executable, "-m", "bhavkit.cli", "query",
     "--db-path", str(WORK / "bhavkit.duckdb"), "--template", "table_counts"],
    capture_output=True, text=True, env=dict(os.environ),
)
print(proc.stdout)
assert proc.returncode == 0 and "bhav_daily" in proc.stdout
print("[ok] CLI ran via `python -m bhavkit.cli`")


┏━━━━━━━━━━━━━━━━━━━┳━━━━━━┓
┃ table             ┃ rows ┃
┡━━━━━━━━━━━━━━━━━━━╇━━━━━━┩
│ bhav_daily        │ 6    │
│ index_daily       │ 0    │
│ fo_daily          │ 0    │
│ deliverable_daily │ 0    │
│ equity_master     │ 0    │
└───────────────────┴──────┘

[ok] CLI ran via `python -m bhavkit.cli`


## Running this workbook

```bash
# from the repo root (uses the editable install created by `uv sync`)
uv run --with jupyter --with ipykernel \
    jupyter nbconvert --to notebook --execute --inplace workbook/bhavkit.ipynb

# or interactively
uv run --with ipykernel jupyter notebook workbook/bhavkit.ipynb
```

All state lives under the temp dir printed in section 1. From here, try the real thing:

```bash
uv run bhavkit init
uv run bhavkit update --start 2024-01-01 --end 2024-01-31 --datasets cm,idx,fo,deliv
uv run bhavkit report --month 2024-01
uv run bhavkit query
```